# Week 7 — NYTimes API & Web Scraping (part 1)
**Name: Craig**
**Date: October 12, 2026**
-

In [2]:
# first we import everything we need for the task at hand 

import requests   # this gives us the ability to request data from remote websites
from time import sleep  # this gives us the ability to implement time delays in our code
from getpass import getpass # this is a password-protection feature for python notebooks

key = getpass("Paste your API key here!")

Paste your API key here! ········


In [50]:
query = '"A.I. Slop"'    # This is the search term we are looking for in any of the metadata

begin_date = "19000101"   # this is the begindate for our search (September 1 2026)
end_date = "20260930"     # this is the enddate for our search (September 30 2026)

sort = "oldest"

url = f"https://api.nytimes.com/svc/search/v2/articlesearch.json?q={query}&begin_date={begin_date}&end_date={end_date}&api-key={key}&sort={sort}"
response = requests.get(url)

print(response.status_code)  # 200

200


Pulling the actual documents out of the API response:
- first we need to parse the response as JSON, then we need to find the key that contains the article info

In [51]:
parsed_nytimes_response = response.json()  # interpret response as Python-readable JSON
print(parsed_nytimes_response.keys())      # tells me what keys are in the raw response
print(parsed_nytimes_response['response'].keys())  # tells me what keys are in the "response" parent
nytimes_docs = parsed_nytimes_response['response']['docs']  # tells me what keys are in the "docs" key, which is a child of "response"
print(len(nytimes_docs))

dict_keys(['status', 'copyright', 'response'])
dict_keys(['docs', 'metadata'])
10


In [52]:
print(nytimes_docs[0]['headline'])
print(nytimes_docs[0]['headline']['main'])
print(nytimes_docs[0]['pub_date'])

{'main': 'H\u200bow Long Will A.I.’s ‘Slop’ Era Last?', 'kicker': 'David Wallace-Wells', 'print_headline': 'A.I. is becoming infamous for the ‘slop’ it’s adding to our online lives. But the technology’s real future is still being written.'}
H​ow Long Will A.I.’s ‘Slop’ Era Last?
2024-07-24T16:46:45Z


In [56]:
print(parsed_nytimes_response['response']['metadata'])

{'hits': 124, 'offset': 0, 'time': 210}


# Web scraping with BeautifulSoup!

Below are the libraries i need to import before I begin web scraping

In [57]:
import csv
import json
import os # operating system library, let's us do things like making new folders
import requests
from time import sleep
from bs4 import BeautifulSoup # the scraping library!

Below is some example HTML that we can traverse using the BeautifulSoup functions!

In [60]:
html = """
<html>
  <body>
    <h1>Adeline Ravoux</h1>
    <p class="artist">Vincent van Gogh</p>
    <p class="date">1890</p>
    <a href="/art/1958.31">Artwork page</a>
  </body>
</html>
"""

soup = BeautifulSoup(html, "html.parser")  # html.parser is the built-in parser that comes with python


In [62]:
title = soup.find("h1")  # find the first element that uses the h1 tag
print(title)
print(title.get_text())

<h1>Adeline Ravoux</h1>
Adeline Ravoux


In [63]:
paragraph_elements = soup.find_all("p")
print(paragraph_elements)

[<p class="artist">Vincent van Gogh</p>, <p class="date">1890</p>]


In [67]:
artist = soup.find("p", class_="artist")  #underscore after class, because it's a reserved word in python
print(artist.get_text())
date = artist.find_next_sibling() # finds the next element that is a "sibling" of the referenced element
print(date.get_text())

Vincent van Gogh
1890


# Understanding "robots" permissions

We can view the permissions that a website grants to do things like automatically access data with a "bot" by looking at its `robots.txt` file. It always lives at the robots.txt file in the root directory of a domain (for instance, https://www.clevelandart.org/robots.txt)

We can see that the cleveland art museum robots policy allows for any user agent to access most of the site, with a 10 second Crawl delay:

```
# * 
User-agent: * 
Crawl-delay: 10 
Disallow: /membership 
Disallow: /orders 
Disallow: /api 
Disallow: /errors 
Disallow: /404 
Disallow: /500 
```

In [69]:
headers = {"User-Agent": "INFO664-class-project (educational)"}   # you can call this whatever you want, it shows up on the scraped website's logs as who you are (ie not a Google/AI bot or web browser)

os.makedirs("scraped_pages", exist_ok=True)  # make a new folder to hold our scraped website HTML data. exist_ok = True means, if the directory already exists, do nothing and don't return an error

def fetch_page(url, filename):  # declaring a new function, which takes a URL to be scraped, and a destination filename, as arguments
    if os.path.exists(filename):  # check to see if file already exists, if so, don't download it from the web, load it instead
        with open(filename, "r", encoding="utf-8") as file:
            return file.read()  # in this case, return provides the read file data, and escapes from the function, ignoring everything below

    response = requests.get(url, headers=headers) # grab the file, using our pre-defined headers as the header data

    if not response.ok: # if the response reports an error
        print(url, "failed with status code", response.status_code) # tell us why it failed
        return None # return a "None" object, since there's nothing usable in the response, and escape from this function
    with open(filename, "w",  encoding="utf-8") as file:
        file.write(response.text)

    sleep(10) # wait 10 seconds, since our robots.txt file asks for a 10 second Crawl-delay
    return response.text

In [73]:
url = "https://www.clevelandart.org/art/1942.1065"

html = fetch_page(url, "scraped_pages/1942_1065.html")
soup = BeautifulSoup(html, "html.parser")

print(len(html))


153826


In [71]:
artwork_details = soup.find(id="artwork-details-container")
print(len(artwork_details))

14


In [82]:
children = artwork_details.find_all(True, recursive=False)

for index, child in enumerate(children):
    print(index, child.name, "|", child.get_text())

0 h2 | Details / Information for Romaine Lacaux 
1 div | Romaine Lacaux
2 div | 1864
3 div | Pierre-Auguste Renoir(French, 1841–1919)
4 div | CultureFrance, 19th century
5 div | Mediumoil on fabric
6 div | MeasurementsFramed: 106.7 x 89.2 x 8.9 cm (42 x 35 1/8 x 3 1/2 in.); Unframed: 81.3 x 65 cm (32 x 25 9/16 in.)
7 div | Credit LineGift of the Hanna Fund 1942.1065
8 div | Public DomainYou can copy, modify, and distribute this work, all without asking permission. Learn more about CMA's Open Access Initiative.
9 div | Location222 Impressionism & Post-Impressionism
10 div | ArtLens AppShareDownloadPrint
11 div | ?Did You Know?The son of a poor tailor from Limoges, Renoir began his artistic career at age 13 as an apprentice to a porcelain painter.
12 div | DescriptionThis painting may be Renoir's earliest signed canvas. Its sensitive display of color and light communicates an ideal of delicate, youthful beauty. The luminous tones of the background drapery and of the child's white blouse 